# ESG SHAP by Sector
Phân tích SHAP theo `sector` để tìm driver theo nhóm ngành.

In [ ]:
import json, pickle, pandas as pd, numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import shap

with open('outputs/models/xgb.pkl','rb') as f:
    model = pickle.load(f)
labels = pd.read_csv('data/labels/financials.csv')

mapped_file = next(Path('data/mapped').glob('*.mapped.jsonl'))
rows = [json.loads(l) for l in open(mapped_file, 'r', encoding='utf-8') if l.strip()]
df = pd.DataFrame(rows)
X = df.pivot_table(index=[], columns='schema_field', values='value', aggfunc='last').reset_index(drop=True)
X = X.select_dtypes(include='number').fillna(0)

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X)

sector = labels.get('sector', pd.Series(['Banking']*len(X)))
if len(sector) != len(X):
    sector = pd.Series(['Banking']*len(X))

abs_shap = np.abs(shap_values)
feat_names = X.columns.tolist()

stats = []
for s in pd.Series(sector).unique():
    idx = (pd.Series(sector)==s).values
    if idx.sum()==0: continue
    mean_abs = abs_shap[idx].mean(axis=0)
    top_idx = np.argsort(-mean_abs)[:15]
    for i in top_idx:
        stats.append({'sector': s, 'feature': feat_names[i], 'mean_abs_shap': float(mean_abs[i])})
stats_df = pd.DataFrame(stats).sort_values(['sector','mean_abs_shap'], ascending=[True, False])
display(stats_df.head(30))

pick_sector = stats_df['sector'].iloc[0]
top10 = stats_df[stats_df['sector']==pick_sector].head(10)
plt.figure()
plt.barh(top10['feature'][::-1], top10['mean_abs_shap'][::-1])
plt.title(f'Top drivers (mean |SHAP|) – {pick_sector}')
plt.xlabel('mean |SHAP|')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()
